In [ ]:
from pathlib import Path
import sys

SERVER_PROJECT_ROOT = Path('~/workspace/paper_research').expanduser().resolve()
if str(SERVER_PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(SERVER_PROJECT_ROOT))

from utils.notebook_bootstrap import bootstrap_notebook

PROJECT_ROOT, DIR = bootstrap_notebook(preferred_root=SERVER_PROJECT_ROOT)


In [1]:

import torch
import os, sys
import polars as pl


sys.path.insert(0, os.path.abspath(".."))


MAC_DIR = '/Users/igwanhyeong/PycharmProjects/paper_research/'
WINDOW_DIR = 'C:/Users/USER/PycharmProjects/paper_research/'

if sys.platform == 'win32':
    DIR = WINDOW_DIR
    print(torch.cuda.is_available())
    print(torch.cuda.device_count())
    print(torch.version.cuda)
    print(torch.__version__)
    print(torch.cuda.get_device_name(0))
    print(torch.__version__)
else:
    DIR = MAC_DIR

In [2]:
intermittent_df = pl.read_parquet(DIR + 'sample_data/intermittent_df.parquet')
intermittent_df

demand_dt,demand_qty,seq,oper_part_no
i64,f64,u32,str
201811,2.0,1,"""0001-1002"""
201812,0.0,2,"""0001-1002"""
201813,0.0,3,"""0001-1002"""
201814,0.0,4,"""0001-1002"""
201815,0.0,5,"""0001-1002"""
…,…,…,…
202602,0.0,311,"""ZZ90239"""
202603,0.0,312,"""ZZ90239"""
202604,0.0,313,"""ZZ90239"""


### 1. 기존 Mark Selection

In [3]:
from utils.mark_utils import make_marks_intermittent_default

origin_marked_df = make_marks_intermittent_default(intermittent_df, K=4, p1=0.99, p2=0.999)

origin_marked_df.group_by("mark").len().sort("mark")
origin_marked_df.select(pl.col("delta_t")).describe()

[0.5]
[1.0986122886681098]
.when([(col("z")) <= (dyn float: 1.0986122886681098)]).then(dyn int: 0).otherwise(dyn int: 1).clip([dyn int: 0, dyn int: 1])


statistic,delta_t
str,f64
"""count""",242888.0
"""null_count""",0.0
"""mean""",28.02348
"""std""",43.228727
"""min""",0.0
"""25%""",1.0
"""50%""",3.0
"""75%""",47.0
"""max""",313.0


### 2. 균등 빈도 기반 Mark Selection

In [4]:
from utils.mark_utils import make_marks_even_quantile

even_marked_df = make_marks_even_quantile(intermittent_df, K = 4)

even_marked_df.group_by('mark').len().sort('mark')
even_marked_df.select(pl.col('delta_t')).describe()

statistic,delta_t
str,f64
"""count""",242888.0
"""null_count""",0.0
"""mean""",28.02348
"""std""",43.228727
"""min""",0.0
"""25%""",1.0
"""50%""",3.0
"""75%""",47.0
"""max""",313.0


In [5]:
print(origin_marked_df.select('mark').describe())
print(even_marked_df.select('mark').describe())


shape: (9, 2)
┌────────────┬──────────┐
│ statistic  ┆ mark     │
│ ---        ┆ ---      │
│ str        ┆ f64      │
╞════════════╪══════════╡
│ count      ┆ 242888.0 │
│ null_count ┆ 0.0      │
│ mean       ┆ 0.390641 │
│ std        ┆ 0.51179  │
│ min        ┆ 0.0      │
│ 25%        ┆ 0.0      │
│ 50%        ┆ 0.0      │
│ 75%        ┆ 1.0      │
│ max        ┆ 3.0      │
└────────────┴──────────┘
shape: (9, 2)
┌────────────┬──────────┐
│ statistic  ┆ mark     │
│ ---        ┆ ---      │
│ str        ┆ f64      │
╞════════════╪══════════╡
│ count      ┆ 242888.0 │
│ null_count ┆ 0.0      │
│ mean       ┆ 1.5      │
│ std        ┆ 1.118036 │
│ min        ┆ 0.0      │
│ 25%        ┆ 1.0      │
│ 50%        ┆ 2.0      │
│ 75%        ┆ 2.0      │
│ max        ┆ 3.0      │
└────────────┴──────────┘


In [9]:
even_marked_df.select('mark').describe()

statistic,mark
str,f64
"""count""",242888.0
"""null_count""",0.0
"""mean""",1.5
"""std""",1.118036
"""min""",0.0
"""25%""",1.0
"""50%""",2.0
"""75%""",2.0
"""max""",3.0


In [ ]:
# validation 단계에서 다음 샘플의 분포가 어떻게 될건지를 알아보는게 중요. 분포 공유해야됨.
# RMTPP Regression 학습 검증 및 Loss 기반 검증
# 이후 Evaluation 방법론 공유.